## META | difficulty 1 | Objectives

This final bundle turns common failure modes into visible patterns instead of vague warnings.

Focus:

- wrong sign in wraparound
- wrong root or wrong zeta
- wrong BO / NO comparison
- missing final scaling
- wrong mental model for the Kyber modulus


## MANDATORY | difficulty 3 | Bad Outputs Have Fingerprints

Debugging NTTs is easier when you stop staring at the final vector as one blob.
Each common mistake leaves a characteristic fingerprint:

- wrong sign flips specific wrapped slots
- wrong order makes a correct value set appear shuffled
- missing `n^-1` keeps the shape but scales everything wrong
- wrong zeta corrupts local pair structure early


In [ ]:
# MANDATORY | difficulty 3 | See Four Failure Modes Side By Side

from IPython.display import display

from ntt_learning.toy_ntt import (
    fast_intt_psi_gs_trace,
    fast_ntt_psi_ct_trace,
    forward_ntt_psi,
    negacyclic_reduce,
    schoolbook_convolution,
)
from ntt_learning.visuals import plot_vector_comparison

signal = [1, 2, 3, 4]
forward_trace = fast_ntt_psi_ct_trace(signal, 7681, 1925)
inverse_trace = fast_intt_psi_gs_trace(forward_trace.raw_output, 7681, 1925)

raw = schoolbook_convolution([1, 2, 3, 4], [5, 6, 7, 8])
wrong_sign = [raw[0] + raw[4], raw[1] + raw[5], raw[2] + raw[6], raw[3]]
wrong_order = list(forward_trace.raw_output)
wrong_scale = list(inverse_trace.raw_output)
wrong_root = forward_ntt_psi(signal, 7681, 3383)

print("wrong sign fold:", wrong_sign)
print("correct sign fold:", negacyclic_reduce(raw, n=4))
print("wrong BO-vs-NO comparison:", wrong_order)
print("correct NO output:", forward_trace.normal_order_output)
print("missing final scaling:", wrong_scale)
print("wrong root in direct transform:", wrong_root)
display(
    plot_vector_comparison(
        wrong_sign,
        negacyclic_reduce(raw, n=4),
        left_label="wrong_sign",
        right_label="correct_sign",
        title="Wrong sign vs correct negacyclic fold",
    )
)
display(
    plot_vector_comparison(
        wrong_order,
        forward_trace.normal_order_output,
        left_label="wrong_order",
        right_label="correct_order",
        title="Wrong BO/NO comparison",
    )
)
display(
    plot_vector_comparison(
        wrong_scale,
        inverse_trace.scaled_output,
        left_label="missing_scale",
        right_label="correct_scale",
        title="Missing scale vs corrected inverse",
    )
)


In [ ]:
# MANDATORY | difficulty 3 | Interactive Failure Picker

import ipywidgets as widgets
from IPython.display import display

from ntt_learning.toy_ntt import fast_intt_psi_gs_trace, fast_ntt_psi_ct_trace, forward_ntt_psi
from ntt_learning.visuals import plot_vector_comparison

forward_trace = fast_ntt_psi_ct_trace([1, 2, 3, 4], 7681, 1925)
inverse_trace = fast_intt_psi_gs_trace(forward_trace.raw_output, 7681, 1925)

failures = {
    "wrong_order": list(forward_trace.raw_output),
    "correct_order": list(forward_trace.normal_order_output),
    "missing_scale": list(inverse_trace.raw_output),
    "scaled": list(inverse_trace.scaled_output),
    "wrong_root": forward_ntt_psi([1, 2, 3, 4], 7681, 3383),
}
references = {
    "wrong_order": list(forward_trace.normal_order_output),
    "correct_order": list(forward_trace.normal_order_output),
    "missing_scale": list(inverse_trace.scaled_output),
    "scaled": list(inverse_trace.scaled_output),
    "wrong_root": list(forward_ntt_psi([1, 2, 3, 4], 7681, 1925)),
}

def preview(mode="wrong_order"):
    print(mode, "->", failures[mode])
    display(
        plot_vector_comparison(
            failures[mode],
            references[mode],
            left_label=mode,
            right_label="reference",
            title=f"{mode} compared with the correct reference",
        )
    )

display(widgets.interact(preview, mode=sorted(failures)))


## MANDATORY | difficulty 2 | Retrieval Check

1. Which mistake keeps the general shape of the inverse output but leaves every entry too large by a shared factor?
2. Which mistake often disappears once you apply the correct BO -> NO reorder?
3. Which mistake shows up earliest in local pair traces rather than only at the very end?


In [ ]:
# FACULTATIVE | difficulty 4 | Optional: Trace Rows For Debugging

from ntt_learning.toy_ntt import fast_ntt_psi_ct_trace, stage_rows

trace = fast_ntt_psi_ct_trace([1, 2, 3, 4], 7681, 1925)
for stage in trace.stages:
    print("stage", stage.stage_index)
    for row in stage_rows(stage):
        print(row)


## META | difficulty 1 | Next Notebook

Next notebook: `lab.ipynb`
